In [1]:
with open("harrypotter.txt", "r", encoding="ISO-8859-1")as f:
    raw_text= f.read()

print("Total number of characters:", len(raw_text))
print(raw_text[:99])

Total number of characters: 442745
Harry Potter and the Sorcerer's Stone 

CHAPTER ONE 

THE BOY WHO LIVED 

Mr. and Mrs. Dursley, of 


In [2]:
import re

text= "Hello, How are you?"
result= re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'How', ' ', 'are', ' ', 'you?']


In [3]:
result= re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'How', ' ', 'are', ' ', 'you?']


In [4]:
result= [item for item in result if item.strip()]
print(result)

['Hello', ',', 'How', 'are', 'you?']


In [5]:
text= "Hello, How are you?-- I am good!"
result= re.split(r'([,.:?_!"()\']|--|\s)',text)
result= [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'How', 'are', 'you', '?', '--', 'I', 'am', 'good', '!']


In [6]:
preprocessed_text= re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed_text= [item.strip() for item in preprocessed_text if item.strip()]
print(preprocessed_text[:30])

['Harry', 'Potter', 'and', 'the', 'Sorcerer', "'", 's', 'Stone', 'CHAPTER', 'ONE', 'THE', 'BOY', 'WHO', 'LIVED', 'Mr', '.', 'and', 'Mrs', '.', 'Dursley', ',', 'of', 'number', 'four', ',', 'Privet', 'Drive', ',', 'were', 'proud']


In [7]:
print(len(preprocessed_text))

102341


Step-2 Create Token IDs

In [8]:
all_words= sorted(set(preprocessed_text))
vocab_size= len(all_words)
print(vocab_size)

6818


In [9]:
vocab= {token: integer for integer, token in enumerate(all_words)}

In [10]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i>=50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('-', 6)
('--', 7)
('-Harry', 8)
('-I', 9)
('-all', 10)
('-an', 11)
('-and', 12)
('-bodied', 13)
('-can', 14)
('-face', 15)
('-for', 16)
('-he', 17)
('-here', 18)
('-how', 19)
('-lessons', 20)
('-please', 21)
('-the', 22)
('-then', 23)
('-they', 24)
('-well', 25)
('-you', 26)
('.', 27)
('0', 28)
('1', 29)
('1473', 30)
('1637', 31)
('17', 32)
('1709', 33)
('1945', 34)
('2', 35)
('3', 36)
('31', 37)
('382', 38)
('4', 39)
('90', 40)
(':', 41)
(';', 42)
('?', 43)
('A', 44)
('AAAAAAAAAARGH', 45)
('AAAARGH', 46)
('ALBUS', 47)
('ALBUS-', 48)
('ALL', 49)
('ALLEY', 50)


In [11]:
class SimpleTokenizerV1:
    def __init__(self,vocab):
        self.str_to_int= vocab
        self.int_to_str= {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed= re.split(r'([.,;!()"_:()\']|--|\s)', text)
        preprocessed= [item.strip() for item in preprocessed if item.strip()]
        ids= [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self,ids):
        dtext= " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations

        dtext= re.sub(r'\s+ ([.,?!"()\'])', r'\1', text)
        return dtext


In [12]:
tokenizer= SimpleTokenizerV1(vocab)
text= """"It's the last he painted, you know," ALBUS said."""
ids= tokenizer.encode(text)
print(ids)

[1, 688, 2, 5179, 6124, 3958, 3574, 4583, 5, 6800, 3928, 5, 1, 47, 5191, 27]


In [13]:
decode_text= tokenizer.decode(ids)
print(decode_text)

"It's the last he painted, you know," ALBUS said.


In [14]:
text= "Hello, would you like some tea?"
ids= tokenizer.encode(text)
print(ids)

KeyError: 'tea?'

Adding Special Context Tokens if in case thr word isn't present in vocab

In [15]:
all_tokens= sorted(list(set(preprocessed_text)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab= {token:i for i,token in enumerate(all_tokens)}


In [16]:
len(vocab.items())

6820

In [17]:
for i,item in enumerate(list(vocab.items())[-5:]):
    print(item)

('zooming', 6815)
('\x93', 6816)
('\x96', 6817)
('<|endoftext|>', 6818)
('<|unk|>', 6819)


In [18]:
class SimpleTokenizerV2:
    def __init__(self,vocab):
        self.str_to_int= vocab
        self.int_to_str= {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed= re.split(r'([;:,.()!?_"\']|--|\s)',text)
        preprocessed= [item.strip() for item in preprocessed if item.strip()]
        preprocessed= [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids= [self.str_to_int[s] for s  in preprocessed]

        return ids

    def decode(self,ids):
        dtext= " ".join([self.int_to_str[i] for i in ids])
        dtext= re.sub(r'\s+([;:,.()!?_"\'])', r'\1',dtext)
        return dtext

In [19]:
tokenizer= SimpleTokenizerV2(vocab)
text1= "Hello, do you like tea?"
text2= "In the sunlit terraces of the palace"
text= " <|endoftext|> ".join((text1,text2))

print(text)



Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace


In [20]:
ids= tokenizer.encode(text)
print(ids)

[616, 5, 2693, 6800, 4041, 6072, 43, 6818, 677, 6124, 6819, 6819, 4488, 6124, 4589]


In [21]:
tokenizer.decode(ids)

'Hello, do you like tea? <|endoftext|> In the <|unk|> <|unk|> of the palace'

BYTE PAIR ENCODING

In [22]:
! pip3 install tiktoken


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.13.0


In [24]:
tokenizer= tiktoken.get_encoding("gpt2")

In [25]:
text= ("Hello, do you like some tea? <|endoftext|> In the sunlit terraces of someunkownPlace.")

integers= tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 617, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 2954, 593, 27271, 13]


In [26]:
strings= tokenizer.decode(integers)
print(strings)

Hello, do you like some tea? <|endoftext|> In the sunlit terraces of someunkownPlace.


In [27]:
integers= tokenizer.encode("@~ijhggb")
print(integers)

strings= tokenizer.decode(integers)
print(strings)

[31, 93, 2926, 71, 1130, 65]
@~ijhggb


CREATING INPUT- TARGET PAIRS

In [28]:
with open("the-verdict.txt") as f:
    raw_text= f.read()

enc_text= tokenizer.encode(raw_text)
print(len(enc_text))


5145


In [29]:
enc_sample= enc_text[50:]

In [30]:
context_size=4 #length of the input
x= enc_sample[:context_size]
y= enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y: {y}")

x: [290, 4920, 2241, 287]
y: [4920, 2241, 287, 257]


In [31]:
for i in range(1, context_size+1):
    context= enc_sample[:i]
    desired= enc_sample[i]
    print(context,"---->",desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [32]:
for i in range(1,context_size+1):
    context= enc_sample[:i]
    desired= enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [33]:
! pip3 install torch



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self,txt, tokenizer,max_length,stride):
        self.input_ids=[]
        self.target_ids=[]

        #tokenize the dataset
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        for i in range(0, len(token_ids)-max_length, stride):
            input_chunk=  token_ids[i:i+max_length]
            target_chunk= token_ids[i+1: i+max_length+1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self,idx):
        return self.input_ids[idx], self.target_ids[idx]


In [35]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):

    #Initialize the tokenizer
    tokenizer= tiktoken.get_encoding("gpt2")

    #Create Dataset
    dataset= GPTDatasetV1( txt, tokenizer,max_length,stride)

    # Create Dataloader
    dataloader= DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers= num_workers)

    return dataloader



In [36]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text= f.read()

In [37]:
import torch
print("Pytorch version:", torch.__version__)

dataloader= create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter= iter(dataloader)
first_batch= next(data_iter)
print(first_batch)

Pytorch version: 2.12.0+cpu
[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [38]:
dataloader= create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter= iter(dataloader)
inputs,targets= next(data_iter)
print("Inputs:\n", inputs)
print("Targets:\n", targets)


Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


 VECTOR EMBEDDINGS


In [39]:
import gensim.downloader as api
model= api.load("word2vec-google-news-300")

[================================================--] 97.7% 1623.9/1662.8MB downloaded

EXAMPLE OF WORD AS VECTOR

In [42]:
word_vector= model
#Let us look at vector embedding for the word computer
print(word_vector['computer'])

[ 1.07421875e-01 -2.01171875e-01  1.23046875e-01  2.11914062e-01
 -9.13085938e-02  2.16796875e-01 -1.31835938e-01  8.30078125e-02
  2.02148438e-01  4.78515625e-02  3.66210938e-02 -2.45361328e-02
  2.39257812e-02 -1.60156250e-01 -2.61230469e-02  9.71679688e-02
 -6.34765625e-02  1.84570312e-01  1.70898438e-01 -1.63085938e-01
 -1.09375000e-01  1.49414062e-01 -4.65393066e-04  9.61914062e-02
  1.68945312e-01  2.60925293e-03  8.93554688e-02  6.49414062e-02
  3.56445312e-02 -6.93359375e-02 -1.46484375e-01 -1.21093750e-01
 -2.27539062e-01  2.45361328e-02 -1.24511719e-01 -3.18359375e-01
 -2.20703125e-01  1.30859375e-01  3.66210938e-02 -3.63769531e-02
 -1.13281250e-01  1.95312500e-01  9.76562500e-02  1.26953125e-01
  6.59179688e-02  6.93359375e-02  1.02539062e-02  1.75781250e-01
 -1.68945312e-01  1.21307373e-03 -2.98828125e-01 -1.15234375e-01
  5.66406250e-02 -1.77734375e-01 -2.08984375e-01  1.76757812e-01
  2.38037109e-02 -2.57812500e-01 -4.46777344e-02  1.88476562e-01
  5.51757812e-02  5.02929

In [43]:
print(word_vector['cat'].shape)

(300,)


SIMILAR WORDS

**KING+ WOMAN-MAN=?**

In [44]:
# Example of using most_Similar
print(word_vector.most_similar(positive=['king','woman'], negative=['man'], topn=10))

[('queen', 0.7118193507194519), ('monarch', 0.6189674139022827), ('princess', 0.5902431011199951), ('crown_prince', 0.5499460697174072), ('prince', 0.5377321839332581), ('kings', 0.5236844420433044), ('Queen_Consort', 0.5235945582389832), ('queens', 0.518113374710083), ('sultan', 0.5098593235015869), ('monarchy', 0.5087411403656006)]


# Let us Check the similarity b/wa few pair of words

In [46]:
# example of calculating similarity
print(word_vector.similarity('woman','man'))
print(word_vector.similarity('king','queen'))
print(word_vector.similarity('uncle','aunt'))
print(word_vector.similarity('nephew','niece'))
print(word_vector.similarity('water', 'paper'))

0.76640123
0.6510956
0.7643474
0.7594368
0.114080824


In [47]:
print(word_vector.most_similar("tower", topn=5))

[('towers', 0.8531750440597534), ('skyscraper', 0.6417425870895386), ('Tower', 0.639177143573761), ('spire', 0.594687819480896), ('responded_Understood_Atlasjet', 0.5931612253189087)]


# Now Let Us see Vector Similarity

In [48]:
import numpy as np

#Words to compare
word1= 'man'
word2= 'woman'

word3= 'semiconductor'
word4='earthworm'

word5='nephew'
word6= 'niece'

# Calculate Vector difference
vector_difference1= model[word1]-model[word2]
vector_difference2= model[word3]-model[word4]
vector_difference3= model[word5]- model[word6]

#Calculate the magnitude of the vector difference
mag1= np.linalg.norm(vector_difference1)
mag2= np.linalg.norm(vector_difference2)
mag3= np.linalg.norm(vector_difference3)

print(mag1)
print(mag2)
print(mag3)

1.727951
5.6670427
1.9557794
